## Install Once

In [183]:
%pip install -q pandas numpy matplotlib requests flexpolyline openpyxl

Note: you may need to restart the kernel to use updated packages.


## Paths

In [195]:
import os
from pathlib import Path
import shutil

def build_folders():
  
    """
    Create the following folders in the current working directory:
    - Data
    - Routes
    - Routes Clean
    - Routes Emissions
    - HBEFA Clean Data
    - HBEFA Data
    """
    project_root      = Path.cwd()            # ← folder where you launched Jupyter
    DATA_DIR          = project_root / "Data"
    ROUTES_DIR        = project_root / "Routes"
    ROUTES_CLEAN_DIR  = project_root / "Routes Clean"
    EMISSIONS_DIR     = project_root / "Routes Emissions"
    HBEFA_CLEAN_DATA_DIR = project_root / "HBEFA Clean Data"
    HBEFA_DATA_DIR = project_root / "HBEFA Data"

    for d in (DATA_DIR, ROUTES_DIR, ROUTES_CLEAN_DIR,
          EMISSIONS_DIR, HBEFA_CLEAN_DATA_DIR, HBEFA_DATA_DIR):
        d.mkdir(exist_ok=True)

def clear_folders():
    """
    Delete (recursively) the project’s output folders if they exist,
    then recreate them empty so downstream code can write into them.

    Folders cleared / recreated, relative to the notebook’s CWD:
      - Data
      - Routes
      - Routes Clean
      - Routes Emissions
      - HBEFA Clean Data
      - HBEFA Data
    """
    project_root        = Path.cwd()
    DATA_DIR            = project_root / "Data"
    ROUTES_DIR          = project_root / "Routes"
    ROUTES_CLEAN_DIR    = project_root / "Routes Clean"
    EMISSIONS_DIR       = project_root / "Routes Emissions"
    ROUTE_SUM_DIR       = project_root / "Routes Segment Emissions"
    HBEFA_CLEAN_DATA_DIR= project_root / "HBEFA Clean Data"
    HBEFA_DATA_DIR      = project_root / "HBEFA Data"

    for d in (DATA_DIR, ROUTES_DIR, ROUTES_CLEAN_DIR,
              EMISSIONS_DIR, ROUTE_SUM_DIR):
        if d.exists():
            shutil.rmtree(d)      # remove the folder and everything in it
        d.mkdir(parents=True, exist_ok=True)




## Generate Coordinates

In [185]:
import requests
import urllib.parse

def get_coordinates(start, end):

    locations = [start, end]
    locations_coords = []

    for location in locations:

        encoded_location = urllib.parse.quote(location)

        headers = {
            'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
        }

        url = f'https://api.openrouteservice.org/geocode/search?api_key=5b3ce3597851110001cf6248b9817ea066594ff4b590e271021b0633&text={encoded_location}&boundary.country=GBR&layers=address,postalcode&size=1'
        call = requests.get(url, headers=headers)

        if call.status_code == 200:
            data = call.json()
            if data.get("features"):
                features = data["features"][0]
                coordinates = features["geometry"]["coordinates"][::-1]
                label = features["properties"]["label"]
                print("Coordinates:", coordinates)
                print("Label:", label)
                locations_coords.append(coordinates)
            else:
                print("No features found in the response.")
        else:
            print("Request failed:", call.status_code, call.reason)

    return locations_coords

## Generate Routes

In [186]:
from pathlib import Path
import json
import requests
import pandas as pd     # still imported because you said you need it later

def get_route(locations_coords, vehicle_type, route_mode,
              departure_time, number_alternatives):
    if not locations_coords or len(locations_coords) < 2:
        raise ValueError("At least two coordinates (origin & destination) are required")

    api_key = "ampQkKtRBHju57U3exZDky2F9oykA1j56mAYKrFjiRg"

    url = (
        f"https://router.hereapi.com/v8/routes?"
        f"apiKey={api_key}&"
        f"origin={locations_coords[0][0]},{locations_coords[0][1]}&"
        f"destination={locations_coords[1][0]},{locations_coords[1][1]}&"
        f"transportMode={vehicle_type}&"
        f"routingMode={route_mode}&"
        f"departureTime={departure_time}&"
        f"return=elevation,polyline,summary,typicalDuration,turnByTurnActions&"
        f"spans=streetAttributes,carAttributes,names,length,duration,baseDuration,"
        f"typicalDuration,maxSpeed,dynamicSpeedInfo&"
        f"alternatives={number_alternatives}"
    )
    response = requests.post(url, json={}, headers={"Content-Type": "application/json"})
    response.raise_for_status()              # let Python throw if we didn’t get 200

    data = response.json()
    (DATA_DIR / "route.txt").write_text(json.dumps(data, indent=2))
    print("JSON written to", (DATA_DIR / 'route.txt').relative_to(project_root))

    # Return the polylines exactly as before
    polylines = [
        section["polyline"]
        for route in data.get("routes", [])
        for section in route.get("sections", [])
        if section.get("polyline")
    ]
    return polylines


## Plot Polylines

In [187]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import flexpolyline as fpl
from pathlib import Path

def plot_polylines(polylines):
    """
    • combined plot  → data_dir/Polyline.png
    • individual     → data_dir/Polyline_route-<n>.png
    """
    DATA_DIR.mkdir(exist_ok=True)

    colors     = ['red', 'blue', 'green', 'orange', 'purple', 'black']
    linestyles = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]

    # ── A) combined figure ───────────────────────────────────────
    fig_c, ax_c = plt.subplots(figsize=(10, 6))

    for idx, encoded in reversed(list(enumerate(polylines, start=1))):
        lats, lons = zip(*[(lat, lon) for lat, lon, *_ in fpl.decode(encoded)])

        colour = colors[(idx - 1) % len(colors)]
        style  = linestyles[(idx - 1) % len(linestyles)]
        z      = 5 + idx

        ax_c.plot(lons, lats, lw=2, linestyle=style, color=colour,
                  label=f'Route {idx}', zorder=z,
                  path_effects=[pe.Stroke(linewidth=5, foreground='white'),
                                pe.Normal()])
        ax_c.plot(lons[0],  lats[0], marker='o', markersize=7,
                  markerfacecolor=colour, markeredgecolor='black', zorder=z+0.5)
        ax_c.plot(lons[-1], lats[-1], marker='x', markersize=9,
                  color=colour, markeredgewidth=2, zorder=z+0.5)

    ax_c.set(title='Decoded Routes from Flexible Polyline',
             xlabel='Longitude', ylabel='Latitude')
    ax_c.axis('equal'); ax_c.grid(True); ax_c.legend()

    comb_path = DATA_DIR / "Polyline.png"
    fig_c.savefig(comb_path, dpi=300, bbox_inches="tight")
    plt.close(fig_c)
    print("✓ combined plot:", comb_path.relative_to(Path.cwd()))

    # ── B) one figure per route ─────────────────────────────────
    for idx, encoded in enumerate(polylines, start=1):
        lats, lons = zip(*[(lat, lon) for lat, lon, *_ in fpl.decode(encoded)])

        colour = colors[(idx - 1) % len(colors)]
        style  = linestyles[(idx - 1) % len(linestyles)]

        fig, ax = plt.subplots(figsize=(7, 5))

        ax.plot(lons, lats, lw=2, linestyle=style, color=colour,
                path_effects=[pe.Stroke(linewidth=5, foreground='white'),
                              pe.Normal()])
        ax.plot(lons[0],  lats[0], marker='o', markersize=7,
                markerfacecolor=colour, markeredgecolor='black')
        ax.plot(lons[-1], lats[-1], marker='x', markersize=9,
                color=colour, markeredgewidth=2)

        ax.set(title=f'Route {idx}', xlabel='Longitude', ylabel='Latitude')
        ax.axis('equal'); ax.grid(True)

        out_path = DATA_DIR / f"Polyline_route-{idx}.png"
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print("✓ saved", out_path.relative_to(Path.cwd()))



## Build Route Data

In [188]:
from pathlib import Path
import json
import pandas as pd

# ────────────────────────────────────────────────────────────────
#  Helper stays the same
# ────────────────────────────────────────────────────────────────
def assign_road_type(road_number):
    if not isinstance(road_number, str) or not road_number:
        return "Local"
    if road_number.startswith("M"):
        return "MW"
    if road_number.startswith("A"):
        digits = road_number[1:]
        if digits.isdigit():
            if len(digits) == 1:
                return "Trunk"
            elif len(digits) == 2:
                return "Distr"
            else:
                return "Local"
    if road_number.startswith("B"):
        return "Access"
    return "n/a"

# ────────────────────────────────────────────────────────────────
#  Cleaning function (unchanged except type hints)
# ────────────────────────────────────────────────────────────────
def clean_route_folder(input_folder: Path, output_dir_name: str = "Routes Clean"):
    input_dir = Path(input_folder)
    output_dir = input_dir.parent / output_dir_name
    output_dir.mkdir(parents=True, exist_ok=True)

    for file_path in input_dir.iterdir():
        if file_path.stem.endswith("_clean"):
            continue
        if file_path.suffix.lower() not in {".csv", ".xlsx", ".xls"}:
            continue

        # Load data
        df = (
            pd.read_csv(file_path)
            if file_path.suffix.lower() == ".csv"
            else pd.read_excel(file_path)
        )

        # Transform
        df["road_type"] = df["road_number"].apply(assign_road_type)
        df_clean = (
            df[["distance_m", "maxSpeed_km_per_h", "trafficSpeed_km_per_h", "road_type"]]
            .rename(
                columns={
                    "distance_m": "distance(m)",
                    "maxSpeed_km_per_h": "speed_limit(km/h)",
                    "trafficSpeed_km_per_h": "vehicle_speed(km/h)",
                }
            )
        )

        # Save
        out_file = output_dir / f"{file_path.stem}_clean{file_path.suffix}"
        (df_clean.to_csv if out_file.suffix == ".csv" else df_clean.to_excel)(
            out_file, index=False
        )
        print("✓ cleaned", out_file.relative_to(input_dir.parent))

# ────────────────────────────────────────────────────────────────
#  Main routine turned into a function with arguments
# ────────────────────────────────────────────────────────────────
def process_meter_data(api_json: Path, routes_dir: Path):
    routes_dir.mkdir(exist_ok=True)
    data = json.loads(api_json.read_text())
    routes = data.get("routes", [])

    for i, route in enumerate(routes, start=1):
        section = route.get("sections", [])[0]
        summary = section.get("summary", {})
        journey_length = summary.get("length", 0)
        df = pd.DataFrame(index=range(1, journey_length + 1))
        df.index.name = "distance_m"

        # save journey duration
        journey_duration = summary.get("typicalDuration", 0)
        if 'journey_durations' not in locals():
            journey_durations = pd.DataFrame(columns=["route_index", "journey_duration"])
        journey_durations = pd.concat(
            [journey_durations, pd.DataFrame({"route_index": [i], "journey_duration": [journey_duration]})],
            ignore_index=True
        )
        out_path = DATA_DIR / f"journey_durations.csv"
        journey_durations.to_csv(out_path, index=False)
        print(f"✓ journey durations saved")

        # -------------- your original STEPS 1–5 unmodified -------------
        df["maxSpeed_km_per_h"] = None
        df["trafficSpeed_km_per_h"] = None
        df["baseSpeed_km_per_h"] = None

        spans = section.get("spans", [])
        cumulative_meter = 0
        for span in spans:
            span_length = span.get("length", 0)
            if span_length <= 0:
                continue
            max_speed = span.get("maxSpeed")
            dyn = span.get("dynamicSpeedInfo", {})
            traffic_speed = dyn.get("trafficSpeed")
            base_speed = dyn.get("baseSpeed")

            max_kmh = max_speed * 3.6 if max_speed is not None else None
            traf_kmh = traffic_speed * 3.6 if traffic_speed is not None else None
            base_kmh = base_speed * 3.6 if base_speed is not None else None

            start = cumulative_meter + 1
            end = cumulative_meter + span_length
            df.loc[start:end, "maxSpeed_km_per_h"] = max_kmh
            df.loc[start:end, "trafficSpeed_km_per_h"] = traf_kmh
            df.loc[start:end, "baseSpeed_km_per_h"] = base_kmh

            cumulative_meter += span_length

        df["road"] = None
        df["road_number"] = None
        df["actions"] = None

        tbt = section.get("turnByTurnActions", [])
        cumulative = 0
        for act in tbt:
            seg_len = act.get("length", 0)
            start = cumulative + 1
            end = cumulative + seg_len
            cumulative += seg_len

            road = act.get("nextRoad", {}).get("name", [{}])[0].get("value", "")
            number = act.get("nextRoad", {}).get("number", [{}])[0].get("value", "")
            desc = act.get("action", "")
            if "direction" in act:
                desc += f" ({act['direction']})"

            df.loc[start:end, "road"] = road
            df.loc[start:end, "road_number"] = number
            if desc:
                df.loc[start, "actions"] = desc

        df["actions"] = df["actions"].shift(-1)

        cols = [
            "maxSpeed_km_per_h",
            "trafficSpeed_km_per_h",
            "baseSpeed_km_per_h",
            "road",
            "road_number",
            "actions",
        ]
        comp = df.fillna("").ne(df.fillna("").shift())[cols].any(axis=1)
        df_compressed = df[comp].copy()

        df_compressed["road_type"] = df_compressed["road_number"].apply(
            assign_road_type
        )

        out_path = routes_dir / f"route_{i}.csv"
        df_compressed.to_csv(out_path, index=True)
        print(f"✓ route {i} saved")



## Clean HBEFA Data

In [189]:
from pathlib import Path
import pandas as pd

# ────────────────────────────────────────────────────────────────
#  1.  Helper: clean a single worksheet (unchanged logic)
# ────────────────────────────────────────────────────────────────
def build_emission_db_pc(excel_file: Path, sheet_name=0) -> pd.DataFrame:
    """
    Reads a sheet formatted like PC.xlsx and returns the tidy DataFrame:

    ['component', 'environment', 'road_type', 'speed_limit',
     'traffic', 'Average_speed', 'Emission_factor']
    """
    df = pd.read_excel(excel_file, sheet_name=sheet_name)

    pieces = df["TrafficSit"].str.split("/", expand=True)
    pieces.columns = ["environment", "road_type", "speed_limit", "traffic"]

    df["environment"] = pieces["environment"]
    df["road_type"]   = pieces["road_type"]

    df["speed_limit"] = pd.to_numeric(pieces["speed_limit"].str.replace(">", ""),
                                       errors="coerce")
    df["traffic"] = pieces["traffic"]

    return (
        df[["Component", "environment", "road_type", "speed_limit",
            "traffic", "V_weighted", "EFA_weighted"]]
        .rename(columns={
            "Component":    "component",
            "V_weighted":   "Average_speed",
            "EFA_weighted": "Emission_factor",
        })
    )

# ────────────────────────────────────────────────────────────────
#  2.  Batch-clean an entire folder
# ────────────────────────────────────────────────────────────────
def clean_all_excels(input_dir=HBEFA_DATA_DIR,
                     output_dir_name="HBEFA Clean Data") -> None:
    """
    Cleans every *.xlsx* (case-insensitive) in *input_dir* and writes the
    results to *output_dir_name* sitting **beside** *input_dir*.
    """
    input_dir   = Path(input_dir)
    output_dir  = input_dir.parent / output_dir_name
    output_dir.mkdir(parents=True, exist_ok=True)

    excel_files = list(input_dir.glob("*.xlsx")) + list(input_dir.glob("*.XLSX"))
    if not excel_files:
        print("No Excel files found in", input_dir)
        return

    for f in excel_files:
        if f.stem.endswith("_clean"):
            continue
        cleaned = build_emission_db_pc(f)
        out_path = output_dir / f"{f.stem}_clean.xlsx"
        cleaned.to_excel(out_path, index=False)
        print("✓", out_path.relative_to(output_dir.parent))




## Estimate Emissions

In [190]:
import pandas as pd
import numpy as np
from pathlib import Path

def assign_emission_factors_to_routes(route_folder: str,
                                      vehicle_type: str,
                                      clean_data_dir: str = HBEFA_CLEAN_DATA_DIR,
                                      output_dir_name: str = 'Routes Emissions'):
    """
    Reads every *_clean.csv in route_folder, and for each row:
      - extracts road_type, speed_limit, and vehicle_speed
      - for each pollutant/component in the vehicle_type EF table, interpolates
        the emission factor based on those properties
      - adds a column per component with its emission factor
    Saves the enriched files to a sibling folder named output_dir_name.
    """
    route_folder = Path(route_folder)
    # Load the EF table once


    ## Special case for each type of vehicle
    NAME_MAP = {
    "car": "PC",       # passenger car
    # add more special cases if needed
    }

    stem = NAME_MAP.get(vehicle_type.lower(), vehicle_type.upper())
    ef_path = Path(clean_data_dir) / f"{stem}_clean.xlsx"  # updated to use stem
    if not ef_path.exists():
        raise FileNotFoundError(f"Emission-factor file not found: {ef_path}")
    ef_df = pd.read_excel(ef_path)
    
    components = ef_df['component'].unique()
    
    # Prepare output directory
    output_dir = route_folder.parent / output_dir_name
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Column names in route files
    sl_col = 'speed_limit(km/h)'
    vs_col = 'vehicle_speed(km/h)'
    
    for route_file in route_folder.glob("*_clean.csv"):
        df_route = pd.read_csv(route_file)
        
        # For each component, compute EF column
        for comp in components:
            def compute_ef(row):
                rt = row['road_type']
                sl = row[sl_col]
                vs = row[vs_col]
                
                # Filter EF table
                subset = ef_df[
                    (ef_df['component'] == comp) &
                    (ef_df['road_type'] == rt) &
                    (ef_df['speed_limit'] == sl)
                ].sort_values('Average_speed')
                
                if subset.empty:
                    # nearest speed_limit fallback
                    avail = ef_df[
                        (ef_df['component'] == comp) &
                        (ef_df['road_type'] == rt)
                    ]['speed_limit'].dropna().unique()
                    if len(avail) == 0:
                        return np.nan
                    nearest = min(avail, key=lambda x: abs(x - sl))
                    subset = ef_df[
                        (ef_df['component'] == comp) &
                        (ef_df['road_type'] == rt) &
                        (ef_df['speed_limit'] == nearest)
                    ].sort_values('Average_speed')
                
                speeds = subset['Average_speed'].to_numpy()
                emis   = subset['Emission_factor'].to_numpy()
                return float(np.interp(vs, speeds, emis, left=emis[0], right=emis[-1]))
            
            df_route[comp] = df_route.apply(compute_ef, axis=1)
        
        # Save enriched route
        out_name = route_file.stem + '_emissions.csv'
        df_route.to_csv(output_dir / out_name, index=False)
        print(f"Saved: {output_dir / out_name}")

def save_segment_emissions_with_summary(route_emissions_folder: str,
                                        output_folder_name: str = "Routes Segment Emissions"):
    """
    Processes each *_clean_emissions.csv in route_emissions_folder:
      1) computes per-segment emissions (g) and saves as *_segment_emissions.csv
      2) aggregates per-route total emissions and writes a combined summary CSV
         named all_routes_summary.csv in the same output folder
    """
    input_dir = Path(route_emissions_folder)
    output_dir = input_dir.parent / output_folder_name
    output_dir.mkdir(parents=True, exist_ok=True)

    summary_list = []

    for file in input_dir.glob("*_clean_emissions.csv"):
        df = pd.read_csv(file)

        # Compute segment distances (m): next distance minus current
        distances = df["distance(m)"].to_numpy()
        segment_m = np.empty_like(distances, dtype=float)
        segment_m[:-1] = distances[1:] - distances[:-1]
        segment_m[-1] = 0  
        segment_km = segment_m / 1000.0

        # Identify pollutant columns (all except metadata)
        metadata = {"distance(m)", "speed_limit(km/h)", "vehicle_speed(km/h)", "road_type"}
        pollutant_cols = [c for c in df.columns if c not in metadata]

        # Calculate emissions in grams per segment
        df_emissions = pd.DataFrame(index=df.index)
        df_emissions["distance(m)"] = df["distance(m)"]
        for col in pollutant_cols:
            df_emissions[col] = df[col] * segment_km

        # Save per-segment emissions file
        out_seg = output_dir / file.name.replace("_clean_emissions.csv", "_segment_emissions.csv")
        df_emissions.to_csv(out_seg, index=False)

        # Get journey duration from data file
        journey_durations = pd.read_csv(DATA_DIR / "journey_durations.csv")

        # Aggregate per-route totals
        totals = {"distance(m)": df_emissions["distance(m)"].iloc[-1]}
        totals["journey_duration(min)"] = journey_durations.loc[journey_durations["route_index"] == int(file.stem.split("_")[1]), "journey_duration"].values[0]/60
        totals.update(df_emissions[pollutant_cols].sum().to_dict())
        route_name = file.stem.replace("_clean_emissions", "")
        record = {"route": route_name, **totals}
        summary_list.append(record)

    # Build combined summary DataFrame and save
    summary_df = pd.DataFrame(summary_list)
    out_summary = output_dir / "all_routes_summary.csv"
    summary_df.to_csv(out_summary, index=False)

    print(f"Saved combined summary: {out_summary}")


## Run Programme

In [196]:
# Route Variables

# Start and End Postcode
start = "EN9 3QH"
end = "W14 9PE"

# Routing characteristics
vehicle_type = "car" # car / ...
route_mode = "fast" # short / fast
departure_time = "2025-02-20T16:00:00+00:00" # 2025-02-20T23:08:00+00:00
number_alternatives = "3" # 0 <= n <= 6

def main(): 
    print("Building folders...")
    # build_folders()
    clear_folders()

    print("Starting coordinate lookup...") 
    locations_coords = get_coordinates(start, end)

    print("Getting route data...")
    polylines = get_route(locations_coords, vehicle_type, route_mode, departure_time, number_alternatives)

    print("Plotting polylines...")
    plot_polylines(polylines)                 # plain axes

    print("Processing meter data...")
    process_meter_data(DATA_DIR / "route.txt", ROUTES_DIR)
    clean_route_folder(ROUTES_DIR) 

    print("Cleaning HBEFA data...")
    clean_all_excels(HBEFA_DATA_DIR, HBEFA_CLEAN_DATA_DIR)

    print("Estimating emissions...")
    assign_emission_factors_to_routes(
    route_folder=ROUTES_CLEAN_DIR,
    vehicle_type="PC",
    clean_data_dir=HBEFA_CLEAN_DATA_DIR,
    )
    save_segment_emissions_with_summary(EMISSIONS_DIR)


main()

Building folders...
Starting coordinate lookup...
Coordinates: [51.672658, 0.006584]
Label: EN9 3QH, Waltham Abbey Civil Parish, England, United Kingdom
Coordinates: [51.488842, -0.205115]
Label: W14 9PE, London, England, United Kingdom
Getting route data...
JSON written to Data/route.txt
Plotting polylines...
✓ combined plot: Data/Polyline.png
✓ saved Data/Polyline_route-1.png
✓ saved Data/Polyline_route-2.png
✓ saved Data/Polyline_route-3.png
✓ saved Data/Polyline_route-4.png
Processing meter data...
✓ journey durations saved
✓ route 1 saved
✓ journey durations saved
✓ route 2 saved
✓ journey durations saved
✓ route 3 saved
✓ journey durations saved
✓ route 4 saved
✓ cleaned Routes Clean/route_2_clean.csv
✓ cleaned Routes Clean/route_3_clean.csv
✓ cleaned Routes Clean/route_1_clean.csv
✓ cleaned Routes Clean/route_4_clean.csv
Cleaning HBEFA data...
✓ HBEFA Clean Data/Check_clean.xlsx
✓ HBEFA Clean Data/PC_clean.xlsx
✓ HBEFA Clean Data/Coach_clean.xlsx
✓ HBEFA Clean Data/MC_clean.xlsx